Good decision. No distraction now—**Day 6 = execution + thinking upgrade**.

I’ll give you **clear tasks only (no over-explanation)**.
You already know the process.

---

# 🚀 DAY 6 — TASKS

---

# 🧠 PART 1 — SQL (Churn Detection)

## 🎯 Problem

Find customers who:

```text
Ordered in a month
BUT did NOT order in the next month
```

---

## 🧾 Input Table: `orders`

```text
customer_id | order_date
```

---

## 🎯 Output

```text
customer_id | last_active_month
```

---

## ⚠️ Rules

* Use `DATE_TRUNC`
* Use `LEAD()`
* Use `MONTHS_BETWEEN`

---

## 🧠 Thinking Hint

```text
Month → next month → gap > 1 OR next_month is NULL
```

---

---

# 🐍 PART 2 — Python (Nested Aggregation)

## 🎯 Problem

```python
data = [
 {"cust":"A","dept":"X","amount":100},
 {"cust":"A","dept":"X","amount":50},
 {"cust":"A","dept":"Y","amount":70},
 {"cust":"B","dept":"X","amount":200},
 {"cust":"B","dept":"Y","amount":100}
]
```

---

## 🎯 Output

```python
{
 "A": {"X":150, "Y":70},
 "B": {"X":200, "Y":100}
}
```

---

## ⚠️ Must Use

```text
Nested .get()
```

---

---

# 🧪 PART 3 — Python (Order-Based Logic)

## 🎯 Problem

```python
nums = [3,1,4,2,5,3,2]
```

---

## 🎯 Task

```text
Find FIRST number that repeats
```

---

## ⚠️ Important

```text
Not most frequent ❌
First duplicate in sequence ✔
```

---

---

# 🧠 PART 4 — Debug Thinking (Real Engineer)

## 🎯 Scenario

```text
Your pipeline suddenly shows DOUBLE revenue
```

---

## 🎯 Task

Write:

```text
3–5 steps you will check first
```

---

---

# 📊 SUBMISSION FORMAT

Reply like:

```text
SQL:
(thinking + query)

Python 1:
(pattern + code)

Python 2:
(thinking + code)

Debug:
(steps)
```

---

---

# ⚡ RULE

```text
Think first → then code
```

---

This is slightly harder than previous days—that’s intentional.

👉 When done, send your solution
I’ll review and push you to **Day 7 (production-level problems)** 💪


# 🧠 PART 1 — SQL (Churn Detection)

## 🎯 Problem

Find customers who:

```text
Ordered in a month
BUT did NOT order in the next month
```

---

## 🧾 Input Table: `orders`

```text
customer_id | order_date
```

---

## 🎯 Output

```text
customer_id | last_active_month
```

---

## ⚠️ Rules

* Use `DATE_TRUNC`
* Use `LEAD()`
* Use `MONTHS_BETWEEN`

---

## 🧠 Thinking Hint

```text
Month → next month → gap > 1 OR next_month is NULL
```

---

---

In [0]:
from pyspark.sql.functions import to_date
sample_orders = [
    (1, "2024-01-15"),
    (1, "2024-02-10"),
    (1, "2024-04-05"),
    (2, "2024-01-20"),
    (2, "2024-03-18"),
    (3, "2024-01-25"),
    (3, "2024-03-01"),
    (4, "2024-05-12")
]
orders_df = spark.createDataFrame(sample_orders, ["customer_id", "order_date"]) \
    .withColumn("order_date", to_date("order_date"))
orders_df.createOrReplaceTempView("orders")
display(orders_df)

customer_id,order_date
1,2024-01-15
1,2024-02-10
1,2024-04-05
2,2024-01-20
2,2024-03-18
3,2024-01-25
3,2024-03-01
4,2024-05-12


In [0]:
%sql

WITH monthly_orders AS (
  SELECT
    customer_id,
    DATE_TRUNC('Month', order_date) AS order_month
  FROM orders
  GROUP BY customer_id, DATE_TRUNC('Month', order_date)
),
churn_candidates AS (
  SELECT
    customer_id,
    order_month,
    LEAD(order_month) OVER (PARTITION BY customer_id ORDER BY order_month) AS next_month
  FROM monthly_orders
)
SELECT
  customer_id,
  order_month AS last_active_month , next_month
FROM churn_candidates
WHERE next_month IS  NULL
   or  MONTHS_BETWEEN(next_month, order_month) > 1

customer_id,last_active_month,next_month
1,2024-02-01T00:00:00.000Z,2024-04-01T00:00:00.000Z
1,2024-04-01T00:00:00.000Z,null
2,2024-01-01T00:00:00.000Z,2024-03-01T00:00:00.000Z
2,2024-03-01T00:00:00.000Z,null
3,2024-01-01T00:00:00.000Z,2024-03-01T00:00:00.000Z
3,2024-03-01T00:00:00.000Z,null
4,2024-05-01T00:00:00.000Z,null


🎯 Problem
data = [
 {"cust":"A","dept":"X","amount":100},
 {"cust":"A","dept":"X","amount":50},
 {"cust":"A","dept":"Y","amount":70},
 {"cust":"B","dept":"X","amount":200},
 {"cust":"B","dept":"Y","amount":100}
]
🎯 Output
{
 "A": {"X":150, "Y":70},
 "B": {"X":200, "Y":100}
}
⚠️ Must Use
Nested .get()

In [0]:
data = [
 {"cust":"A","dept":"X","amount":100},
 {"cust":"A","dept":"X","amount":50},
 {"cust":"A","dept":"Y","amount":70},
 {"cust":"B","dept":"X","amount":200},
 {"cust":"B","dept":"Y","amount":100}
]

# expected output 
# { "A": {"X":150, "Y":70}, "B": {"X":200, "Y":100} }

result = {}

for i in data:
    cust = i['cust']
    dept = i['dept']
    amount = i['amount']
    
    # Get the nested dict for this customer (or empty dict if customer doesn't exist)
    cust_dict = result.get(cust, {})
    
    # Get the current amount for this dept (or 0 if dept doesn't exist)
    current_amount = cust_dict.get(dept, 0)
    
    # Add the new amount
    cust_dict[dept] = current_amount + amount
    
    # Update the result
    result[cust] = cust_dict

print(result)

{'A': {'X': 150, 'Y': 70}, 'B': {'X': 200, 'Y': 100}}


In [0]:
# 🎯 Problem
nums = [3,1,4,2,5,3,2]

result = {}
for i in nums:
    result[i] = result.get(i, 0) + 1
    if result[i] > 1:
        print(f"First duplicate: {i}")
        break  # Stop after finding the first duplicate!
else:
    print("No duplicates found")

First duplicate: 3


In [0]:
seen = set()
for i in nums:
    if i in seen:
        print(f'first duplicate : {i}')
        break
    seen.add(i)

first duplicate : 3


# 📚 Data Structures Cheat Sheet — When to Use What?

---

## 🎯 Quick Decision Guide

| Need | Use | Why |
|------|-----|-----|
| Check if item exists | **Set** | O(1) lookup |
| Remove duplicates | **Set** | Automatic uniqueness |
| Count frequencies | **Dict** | Key → count mapping |
| Group by key | **Dict** | Key → values mapping |
| Maintain order | **List** | Preserves insertion order |
| Allow duplicates | **List** | Can store same value multiple times |
| Need indexing | **List** | Access by position |
| Immutable collection | **Tuple** | Can't be modified |
| Dict key | **Tuple** | Must be immutable |

---

## 🔍 Real-World Examples Below ↓

In [0]:
# ========================================
# 1️⃣ SET - Duplicate Detection & Membership
# ========================================

print("\n=== SET USE CASES ===")

# ✅ Use Case 1: Find duplicates
email_list = ['a@x.com', 'b@x.com', 'a@x.com', 'c@x.com']
seen = set()
duplicates = [email for email in email_list if email in seen or seen.add(email)]
print(f"Duplicates found: {len(email_list) - len(seen)}")

# ✅ Use Case 2: Remove duplicates (preserve uniqueness)
user_ids = [101, 102, 101, 103, 102, 104]
unique_users = list(set(user_ids))
print(f"Unique users: {unique_users}")

# ✅ Use Case 3: Check membership (fast lookup)
active_users = {101, 105, 203, 408}
user_to_check = 101
if user_to_check in active_users:  # O(1) lookup!
    print(f"User {user_to_check} is active")

# ✅ Use Case 4: Set operations (union, intersection, difference)
team_a = {'Alice', 'Bob', 'Charlie'}
team_b = {'Bob', 'Diana', 'Eve'}
print(f"Both teams: {team_a & team_b}")  # Intersection
print(f"Only in A: {team_a - team_b}")   # Difference


# ========================================
# 2️⃣ DICT - Counting & Mapping
# ========================================

print("\n=== DICT USE CASES ===")

# ✅ Use Case 1: Count frequencies
orders = ['pizza', 'burger', 'pizza', 'sushi', 'burger', 'pizza']
order_counts = {}
for order in orders:
    order_counts[order] = order_counts.get(order, 0) + 1
print(f"Order counts: {order_counts}")

# ✅ Use Case 2: Group by key
transactions = [
    {'user': 'A', 'amount': 100},
    {'user': 'B', 'amount': 50},
    {'user': 'A', 'amount': 200}
]
user_totals = {}
for t in transactions:
    user_totals[t['user']] = user_totals.get(t['user'], 0) + t['amount']
print(f"User totals: {user_totals}")

# ✅ Use Case 3: Map one value to another (lookup table)
status_codes = {200: 'OK', 404: 'Not Found', 500: 'Server Error'}
print(f"Status 404 means: {status_codes.get(404, 'Unknown')}")

# ✅ Use Case 4: Cache/memoization
cache = {}
def expensive_calculation(n):
    if n in cache:
        return cache[n]
    result = n ** 2  # Simulate expensive operation
    cache[n] = result
    return result

print(f"Calculated: {expensive_calculation(5)}")
print(f"From cache: {expensive_calculation(5)}")


# ========================================
# 3️⃣ LIST - Ordered Data & Duplicates
# ========================================

print("\n=== LIST USE CASES ===")

# ✅ Use Case 1: Maintain insertion order
login_times = ['9:00 AM', '9:15 AM', '9:30 AM', '9:15 AM']
print(f"Login sequence: {login_times}")  # Order matters!

# ✅ Use Case 2: Allow duplicates (frequency matters)
ratings = [5, 4, 5, 3, 5, 4, 5]
print(f"Average rating: {sum(ratings) / len(ratings):.2f}")

# ✅ Use Case 3: Index-based access
steps = ['Login', 'Select Product', 'Checkout', 'Payment']
print(f"Step 2: {steps[1]}")

# ✅ Use Case 4: Stack operations (LIFO)
undo_stack = []
undo_stack.append('Action 1')
undo_stack.append('Action 2')
last_action = undo_stack.pop()
print(f"Undoing: {last_action}")

# ✅ Use Case 5: Queue operations (FIFO) - use collections.deque for efficiency
queue = ['Task 1', 'Task 2', 'Task 3']
first_task = queue.pop(0)
print(f"Processing: {first_task}")


# ========================================
# 4️⃣ TUPLE - Immutable Data
# ========================================

print("\n=== TUPLE USE CASES ===")

# ✅ Use Case 1: Function returning multiple values
def get_user_info():
    return ('Alice', 28, 'Engineer')  # Tuple

name, age, role = get_user_info()  # Unpacking
print(f"User: {name}, Age: {age}, Role: {role}")

# ✅ Use Case 2: Dictionary keys (must be immutable)
locations = {
    (40.7128, -74.0060): 'New York',
    (34.0522, -118.2437): 'Los Angeles'
}
print(f"Location: {locations[(40.7128, -74.0060)]}")

# ✅ Use Case 3: Constant data (shouldn't change)
RGB_RED = (255, 0, 0)
DATABASE_CONFIG = ('localhost', 5432, 'mydb')
print(f"Red color: {RGB_RED}")


# ========================================
# 🔥 BONUS - Advanced Collections
# ========================================

print("\n=== ADVANCED USE CASES ===")

# ✅ defaultdict - auto-initialize missing keys
from collections import defaultdict

user_orders = defaultdict(list)  # Default value is empty list
user_orders['Alice'].append('Pizza')
user_orders['Alice'].append('Burger')
user_orders['Bob'].append('Sushi')
print(f"Orders: {dict(user_orders)}")

# ✅ Counter - automatic frequency counting
from collections import Counter

votes = ['Alice', 'Bob', 'Alice', 'Charlie', 'Alice', 'Bob']
vote_counts = Counter(votes)
print(f"Vote counts: {vote_counts}")
print(f"Winner: {vote_counts.most_common(1)[0][0]}")

print("\n✅ All examples executed successfully!")


=== SET USE CASES ===
Duplicates found: 1
Unique users: [104, 101, 102, 103]
User 101 is active
Both teams: {'Bob'}
Only in A: {'Charlie', 'Alice'}

=== DICT USE CASES ===
Order counts: {'pizza': 3, 'burger': 2, 'sushi': 1}
User totals: {'A': 300, 'B': 50}
Status 404 means: Not Found
Calculated: 25
From cache: 25

=== LIST USE CASES ===
Login sequence: ['9:00 AM', '9:15 AM', '9:30 AM', '9:15 AM']
Average rating: 4.43
Step 2: Select Product
Undoing: Action 2
Processing: Task 1

=== TUPLE USE CASES ===
User: Alice, Age: 28, Role: Engineer
Location: New York
Red color: (255, 0, 0)

=== ADVANCED USE CASES ===
Orders: {'Alice': ['Pizza', 'Burger'], 'Bob': ['Sushi']}
Vote counts: Counter({'Alice': 3, 'Bob': 2, 'Charlie': 1})
Winner: Alice

✅ All examples executed successfully!


## ⚡ Performance Comparison

### Lookup Speed (Finding an element)

| Operation | List | Set | Dict |
|-----------|------|-----|------|
| Check if exists | O(n) ❌ Slow | O(1) ✅ Fast | O(1) ✅ Fast |
| Example | `x in [1,2,3,...]` | `x in {1,2,3,...}` | `x in {1:a, 2:b,...}` |

---

### When to Convert?

**List → Set**: When you need to check membership many times
```python
# ❌ Bad (checking list 1000 times)
user_ids = [1, 2, 3, ...., 10000]  # List
for order in orders:  # 1000 orders
    if order['user_id'] in user_ids:  # O(n) each time!
        process(order)

# ✅ Good (convert once, then fast lookups)
user_ids = {1, 2, 3, ...., 10000}  # Set
for order in orders:  # 1000 orders
    if order['user_id'] in user_ids:  # O(1) each time!
        process(order)
```

---

### Memory Usage

| Data Structure | Memory | Why |
|----------------|--------|-----|
| List | Medium | Stores values + order |
| Set | Low | Stores values only (no order, no duplicates) |
| Dict | High | Stores keys + values + hash table |
| Tuple | Lowest | Immutable, more compact |

---

## 🧠 Decision Tree

```
Do you need to COUNT how many times each item appears?
│
├─ YES → Use DICT (or Counter)
│
└─ NO → Do you only need to know IF an item exists?
    │
    ├─ YES → Use SET
    │
    └─ NO → Do you need to maintain ORDER or allow DUPLICATES?
        │
        ├─ YES → Use LIST
        │
        └─ NO → Is the data IMMUTABLE/CONSTANT?
            │
            ├─ YES → Use TUPLE
            │
            └─ NO → Use LIST
```

## 🎯 Practice Problems — Choose the Right Data Structure

For each scenario below, think about which data structure is best:

---

### Problem 1: Active User Check
**Scenario**: You have 1 million user IDs in a database. Every second, 10,000 requests come in asking "Is user X active?"

**Which to use?**
* ✅ **Set** → Fast O(1) lookup, no duplicates needed
* ❌ List → O(n) lookup = too slow
* ❌ Dict → Wastes memory (don't need values)

---

### Problem 2: Product Rating Analysis
**Scenario**: You need to calculate average rating, but also see the distribution (how many 5-star, 4-star, etc.)

**Which to use?**
* ❌ Set → Removes duplicates (loses rating counts!)
* ❌ List → Can calculate average, but counting each rating manually is tedious
* ✅ **Dict or Counter** → Stores rating → count mapping

---

### Problem 3: Order Processing Queue
**Scenario**: Orders arrive and must be processed in the exact order received (FIFO)

**Which to use?**
* ❌ Set → No order preserved
* ✅ **List (or deque)** → Maintains insertion order
* ❌ Dict → Overkill (don't need key-value mapping)

---

### Problem 4: GPS Coordinates Cache
**Scenario**: Cache city names by (latitude, longitude) coordinates

**Which to use?**
* ❌ List → Can't use coordinates as index
* ❌ Set → Can't map coordinates to city names
* ✅ **Dict with Tuple keys** → `{(lat, lon): 'City Name'}`

---

### Problem 5: Email Validation
**Scenario**: Check if an email has already been used during registration

**Which to use?**
* ✅ **Set** → Fast lookup, automatic uniqueness
* ❌ List → Slow lookup
* ❌ Dict → Don't need to map email to anything

---

### Problem 6: Event Log with Timestamps
**Scenario**: Log every user action with timestamp, need to replay events in order

**Which to use?**
* ❌ Set → No order, loses duplicate events
* ✅ **List** → Preserves order, allows duplicate actions
* ❌ Dict → Can't have duplicate keys (same timestamp)

---

## 💡 Key Takeaway

> **The right data structure makes the difference between O(n) and O(1) performance!**
>
> In interviews and production:
> * Wrong choice = timeout/slow app
> * Right choice = fast, scalable solution

This is not just “Day 6 done.”
This is where your **thinking level is clearly shifting**—and I can prove it from your file.

---

# 🔍 REAL REVIEW (Based on Your Notebook)

I went through your full notebook  — not just answers, but how you structured things.

---

# 🧠 1. SQL — You’ve Crossed the Beginner Phase

Your churn query:

```sql
LEAD(order_month) OVER (PARTITION BY customer_id ORDER BY order_month)
WHERE next_month IS NULL OR MONTHS_BETWEEN(...) > 1
```

### What this tells me:

```text
You understand sequence-based logic
(not just aggregation anymore)
```

👉 This is a **BIG shift**

Most engineers stay stuck in:

```text
GROUP BY → SUM → done ❌
```

You moved into:

```text
Time-based behavior analysis ✔
```

---

# 🐍 2. Python — Your Core Gap Is FIXING

Your nested `.get()` solution:

```python
cust_dict = result.get(cust, {})
current_amount = cust_dict.get(dept, 0)
```

### What this proves:

```text
You now understand:
- nested structure
- safe access
- accumulation pattern
```

👉 This is EXACTLY what you were struggling with 2–3 days back.

---

# ⚡ 3. Best Part of Your Learning (Very Important)

This section in your notebook:

```python
# Data structure cheat sheet
# when to use set, dict, list, tuple
```

👉 This is **gold**

Why?

Because now you’re not just coding—you’re thinking:

```text
“What structure should I use?”
```

---

# 🧠 THIS is senior thinking:

```text
Problem → choose data structure → then code
```

Not:

```text
Problem → start coding randomly ❌
```

---

# 🧪 4. Duplicate Detection — Strong Understanding

You used BOTH:

### dict approach

```python
result[i] = result.get(i, 0) + 1
```

### set approach

```python
seen = set()
```

👉 And you understand the difference.

---

# 🧠 This is a key milestone:

```text
You now choose approach based on problem
```

---

# ⚠️ 5. One Gap Still Exists (IMPORTANT)

You still didn’t properly document:

```text
Debug thinking (production mindset)
```

This part is weak.

---

## Let me fix it for you (you MUST learn this)

Add this to your notes:

---

# 🔥 DEBUG PLAYBOOK (SAVE THIS)

## 🎯 Problem: Revenue Doubled

### Step 1 — Check duplicates

```sql
SELECT *, COUNT(*)
FROM table
GROUP BY all_columns
HAVING COUNT(*) > 1
```

---

### Step 2 — Check JOIN explosion

```text
1 row joins with multiple rows → duplication
```

---

### Step 3 — Check aggregation logic

```text
SUM applied twice?
Wrong grouping?
```

---

### Step 4 — Check pipeline re-run

```text
Data loaded twice?
Incremental logic broken?
```

---

### Step 5 — Check upstream change

```text
Schema or logic changed?
```

---

👉 Memorize this pattern.

This is what interviewers ask indirectly.

---

# 📊 FINAL EVALUATION

### Your Level Now:

```text
SQL:        ⭐⭐⭐⭐☆ (Strong)
Python:     ⭐⭐⭐☆☆ (Improving fast)
Thinking:   ⭐⭐⭐⭐☆ (Big improvement)
Debugging:  ⭐⭐☆☆☆ (Needs work)
```

---

# 🚀 What Changed in You (IMPORTANT)

Before:

```text
“I know tools but can’t think logic”
```

Now:

```text
“I can break problem → apply pattern → code it”
```

---

# 🔥 Day 6 Verdict

```text
You are officially out of beginner zone.
```

---

# 🚀 NEXT STEP

If you're ready:

👉 Say:

```text
"Start Day 7"
```

Day 7 will push you into:

* Real pipeline scenarios
* Debug-heavy thinking
* Multi-step SQL problems
* Slight pressure-based coding

---

# ⚡ Final Note

What you did today:

```text
You didn’t just solve problems
You built decision-making ability
```

That’s the difference between:

```text
Coder ❌
Engineer ✔
```

---

Say when ready 💪
